# Clinical Trials Data Cleaning & Feature Engineering Pipeline
**Thesis:** Predictive Query Rate Modeling in Clinical Trials  
**Student:** Puneetha Chowdari Modepalli Subramanyam Reddamma  
**Programme:** MSc Data Analytics — Berlin School of Business and Innovation (BSBI)  

---

## Purpose
This notebook loads the raw ClinicalTrials.gov dataset (`ctg-studies.zip`), applies an 11-step cleaning and feature engineering pipeline, and saves the final dataset as `cleaned_clinical_trials.csv`.


### Final Output
- **~50,000 rows** of COMPLETED + TERMINATED interventional clinical trials
- **16 engineered features** ready for ML model training
- Saved to: `cleaned_clinical_trials.csv`


In [1]:
import numpy as np
import pandas as pd

# ============================================================
## Step 0: Imports and Configuration
# ============================================================
INPUT_FILE  = "C:/Users/Puni/Desktop/Thesis/ctg-studies.zip"       # Raw source data
OUTPUT_FILE = "C:/Users/Puni/Desktop/Thesis/cleaned_clinical_trials.csv"  # Final cleaned output

# Display all columns when previewing DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

print("Libraries loaded successfully.")
print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")

Libraries loaded successfully.
Input  : C:/Users/Puni/Desktop/Thesis/ctg-studies.zip
Output : C:/Users/Puni/Desktop/Thesis/cleaned_clinical_trials.csv


## Step 1: Load Raw Data

In [2]:
# pandas can read directly from a zip file that contains a single CSV
df_raw = pd.read_csv(INPUT_FILE)

print(f"Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print()
print("Column names:")
for col in df_raw.columns:
    print(f"  - {col}")

Raw dataset loaded: 119,151 rows, 19 columns

Column names:
  - NCT Number
  - Study Title
  - Study URL
  - Study Status
  - Study Results
  - Conditions
  - Interventions
  - Sponsor
  - Collaborators
  - Sex
  - Age
  - Phases
  - Enrollment
  - Study Type
  - Study Design
  - Start Date
  - Primary Completion Date
  - Completion Date
  - Locations


In [3]:
# Quick look at the raw data
df_raw.head(3)

,NCT Number,Study Title,Study URL,Study Status,Study Results,Conditions,Interventions,Sponsor,Collaborators,Sex,Age,Phases,Enrollment,Study Type,Study Design,Start Date,Primary Completion Date,Completion Date,Locations
0,NCT00768222,Coated VICRYL* Plus Suture Compared to Chinese Silk in S...,https://clinicaltrials.gov/study/NCT00768222,COMPLETED,YES,Breast Cancer,DEVICE: silk suture|DEVICE: VICRYL* Plus suture,"Ethicon, Inc.","Johnson & Johnson Medical, China",ALL,"ADULT, OLDER_ADULT",PHASE4,101.0,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: PARALLEL | ...,2008-10,2009-03,2009-05,"Capital Medical Univ. affiliated Hospital, Beijing, Chin..."
1,NCT03471351,Safety and Efficacy Study of Tenalisib (RP6530) in Combi...,https://clinicaltrials.gov/study/NCT03471351,TERMINATED,NO,Classical Hodgkin Lymphoma,DRUG: Tenalisib|BIOLOGICAL: Pembrolizumab,Rhizen Pharmaceuticals SA,NaN,ALL,"ADULT, OLDER_ADULT",PHASE1,2.0,INTERVENTIONAL,Allocation: NA | Intervention Model: SINGLE_GROUP | Mask...,2018-07-18,2019-02-13,2019-02-13,"University of Chicago, Chicago, Illinois, 60637, United ..."
2,NCT00498251,Prevention of Lung Edema After Thoracic Surgery,https://clinicaltrials.gov/study/NCT00498251,COMPLETED,NO,"Lung Injury, Acute|Thoracotomy|Anesthesia|Intensive Care...",DRUG: inhalation of salbutamol (5 mg)|DRUG: ipratropium,"University Hospital, Geneva",NaN,ALL,"CHILD, ADULT, OLDER_ADULT",NaN,30.0,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: CROSSOVER |...,2004-09,NaN,2007-06,"University Hospital of Geneva, Geneva, CH-1211, Switzerland"


In [4]:
# Null counts in raw data — useful baseline before cleaning
print("=== NULL COUNTS IN RAW DATA ===")
null_summary = df_raw.isnull().sum().reset_index()
null_summary.columns = ["Column", "Null Count"]
null_summary["Null %"] = (null_summary["Null Count"] / len(df_raw) * 100).round(1)
print(null_summary.to_string(index=False))

=== NULL COUNTS IN RAW DATA ===
                 Column  Null Count  Null %
             NCT Number           0     0.0
            Study Title           0     0.0
              Study URL           0     0.0
           Study Status           0     0.0
          Study Results           0     0.0
             Conditions           6     0.0
          Interventions        9957     8.4
                Sponsor           0     0.0
          Collaborators       76108    63.9
                    Sex          60     0.1
                    Age           0     0.0
                 Phases       50634    42.5
             Enrollment        2185     1.8
             Study Type           0     0.0
           Study Design         822     0.7
             Start Date         919     0.8
Primary Completion Date        4393     3.7
        Completion Date        5052     4.2
              Locations        9836     8.3


## Step 2: Filter — INTERVENTIONAL Studies Only

The thesis focuses on interventional clinical trials (drug, device, biological, etc.).  
Observational studies and expanded access programmes are excluded as they do not follow  
the same data management workflows relevant to query rate prediction.


In [5]:
print("=== STUDY TYPE BREAKDOWN (RAW) ===")
print(df_raw["Study Type"].value_counts())
print()

df = df_raw[df_raw["Study Type"].str.upper().str.strip() == "INTERVENTIONAL"].copy()

print(f"After INTERVENTIONAL filter: {len(df):,} rows  "
      f"(dropped {len(df_raw) - len(df):,} non-interventional rows)")

=== STUDY TYPE BREAKDOWN (RAW) ===
Study Type
INTERVENTIONAL     94006
OBSERVATIONAL      24730
EXPANDED_ACCESS      415
Name: count, dtype: int64

After INTERVENTIONAL filter: 94,006 rows  (dropped 25,145 non-interventional rows)


## Step 3: Filter — COMPLETED and TERMINATED Trials Only

For ML model training we need trials with **known outcomes** — i.e., trials that have  
actually finished (completed normally or stopped early).  

Excluding:
- `RECRUITING`, `NOT_YET_RECRUITING`, `ACTIVE_NOT_RECRUITING` — still ongoing, no outcome data
- `UNKNOWN` — status unclear, unreliable for supervised learning
- `WITHDRAWN` — never enrolled participants
- `SUSPENDED` — paused, outcome unknown

Keeping:
- `COMPLETED` — finished per protocol
- `TERMINATED` — stopped early (important signal for query risk models)


In [6]:
print("=== STUDY STATUS BREAKDOWN (INTERVENTIONAL) ===")
print(df["Study Status"].value_counts())
print()

VALID_STATUSES = ["COMPLETED", "TERMINATED"]
df = df[df["Study Status"].isin(VALID_STATUSES)].copy()

print(f"After status filter (COMPLETED + TERMINATED): {len(df):,} rows")

=== STUDY STATUS BREAKDOWN (INTERVENTIONAL) ===
Study Status
COMPLETED                  40945
RECRUITING                 14198
UNKNOWN                    13810
TERMINATED                  9792
ACTIVE_NOT_RECRUITING       6415
NOT_YET_RECRUITING          4626
WITHDRAWN                   3439
ENROLLING_BY_INVITATION      398
SUSPENDED                    383
Name: count, dtype: int64

After status filter (COMPLETED + TERMINATED): 50,737 rows


## Step 4: Filter — Remove Zero Enrollment

Trials with zero enrollment had no participants — they cannot generate data queries.  
Null enrollment values are **kept** at this stage and will be imputed later by phase median.


In [7]:
print(f"Zero enrollment rows  : {(df['Enrollment'] == 0).sum():,}")
print(f"Null enrollment rows  : {df['Enrollment'].isna().sum():,}")
print()

df = df[df["Enrollment"] != 0].copy()

print(f"After removing zero enrollment: {len(df):,} rows")

Zero enrollment rows  : 4
Null enrollment rows  : 1,367

After removing zero enrollment: 50,733 rows


## Step 5: Parse Dates — Fix for Mixed Format (YYYY-MM and YYYY-MM-DD)

**This was the critical bug in the previous version.**

ClinicalTrials.gov stores dates in two formats:
- `YYYY-MM` (e.g., `2008-10`) — most common
- `YYYY-MM-DD` (e.g., `2018-07-18`) — newer entries

Using `pd.to_datetime()` without `format='mixed'` causes pandas to silently fail on `YYYY-MM`  
entries and return `NaT`, which then drops those rows when we filter for non-null dates.  

**Fix:** Use `format='mixed'` to handle both formats correctly.  
**Additional fix:** Use `Primary Completion Date` as a fallback when `Completion Date` is null.


In [8]:
# Parse all three date columns with mixed format support
df["Start Date"]               = pd.to_datetime(df["Start Date"],               format="mixed", errors="coerce")
df["Completion Date"]          = pd.to_datetime(df["Completion Date"],           format="mixed", errors="coerce")
df["Primary Completion Date"]  = pd.to_datetime(df["Primary Completion Date"],   format="mixed", errors="coerce")

print("After format='mixed' parsing:")
print(f"  Non-null Start Date             : {df['Start Date'].notna().sum():,}")
print(f"  Non-null Completion Date        : {df['Completion Date'].notna().sum():,}")
print(f"  Non-null Primary Completion Date: {df['Primary Completion Date'].notna().sum():,}")
print()

# Use Primary Completion Date as fallback where Completion Date is missing
df["Best End Date"] = df["Completion Date"].fillna(df["Primary Completion Date"])
print(f"  Non-null Best End Date (combined): {df['Best End Date'].notna().sum():,}")

After format='mixed' parsing:
  Non-null Start Date             : 50,471
  Non-null Completion Date        : 48,424
  Non-null Primary Completion Date: 48,585

  Non-null Best End Date (combined): 50,257


In [9]:
# Drop rows where Start Date or Best End Date is still null after parsing
before = len(df)
df = df[df["Start Date"].notna() & df["Best End Date"].notna()].copy()
print(f"Dropped {before - len(df):,} rows with unparseable dates.")
print(f"Remaining: {len(df):,} rows")

Dropped 590 rows with unparseable dates.
Remaining: 50,143 rows


In [10]:
# Filter out implausible dates: completion before year 2000 or start before 1950
before = len(df)
df = df[df["Best End Date"].dt.year >= 2000].copy()
df = df[df["Start Date"].dt.year > 1950].copy()
print(f"Dropped {before - len(df):,} rows with implausible dates (pre-2000 completion or pre-1950 start).")
print(f"Remaining: {len(df):,} rows")

Dropped 69 rows with implausible dates (pre-2000 completion or pre-1950 start).
Remaining: 50,074 rows


## Step 6: Feature Engineering — Trial Duration (days)


In [11]:
df["Trial Duration (days)"] = (df["Best End Date"] - df["Start Date"]).dt.days

# Remove zero or negative durations (data errors)
before = len(df)
df = df[df["Trial Duration (days)"] > 0].copy()
print(f"Dropped {before - len(df):,} rows with zero or negative trial duration.")

print()
print("Trial Duration (days) — summary statistics:")
print(df["Trial Duration (days)"].describe().round(1))

Dropped 38 rows with zero or negative trial duration.

Trial Duration (days) — summary statistics:
count    50036.0
mean      1521.1
std       1133.7
min          1.0
25%        730.0
50%       1257.0
75%       2022.0
max      14056.0
Name: Trial Duration (days), dtype: float64


## Step 7: Feature Engineering — Number of Sites

Count the number of site locations listed in the `Locations` column.  
Sites are pipe-separated (`|`) in the raw data.  
Trials with no location data are assigned 0 sites.


In [12]:
def count_sites(location_val):
    """Count the number of pipe-separated locations."""
    if pd.isna(location_val) or str(location_val).strip() == "":
        return 0
    return len(str(location_val).split("|"))

df["Number of Sites"] = df["Locations"].apply(count_sites)

print("Number of Sites — summary statistics:")
print(df["Number of Sites"].describe().round(1))
print()
print("Trials with 0 sites (no location data):", (df["Number of Sites"] == 0).sum())

Number of Sites — summary statistics:
count    50036.0
mean        12.3
std         44.5
min          0.0
25%          1.0
50%          1.0
75%          6.0
max       1452.0
Name: Number of Sites, dtype: float64

Trials with 0 sites (no location data): 1957


## Step 8: Feature Engineering — Intervention Type

Extract the primary intervention type (DRUG, DEVICE, BIOLOGICAL, etc.)  
from the first item in the pipe-separated `Interventions` column.


In [13]:
def extract_intervention_type(intervention_val):
    """Extract the category label (e.g. DRUG, DEVICE) from the first intervention entry."""
    if pd.isna(intervention_val) or str(intervention_val).strip() == "":
        return "UNKNOWN"
    first_item = str(intervention_val).split("|")[0]
    if ":" in first_item:
        return first_item.split(":")[0].upper().strip()
    return "OTHER"

df["Intervention Type"] = df["Interventions"].apply(extract_intervention_type)

print("Intervention Type — value counts:")
print(df["Intervention Type"].value_counts())

Intervention Type — value counts:
Intervention Type
DRUG                   29900
BIOLOGICAL              4936
OTHER                   3606
PROCEDURE               3328
BEHAVIORAL              3058
DEVICE                  2411
RADIATION               1346
DIETARY_SUPPLEMENT       711
DIAGNOSTIC_TEST          429
COMBINATION_PRODUCT      172
GENETIC                  139
Name: count, dtype: int64


## Step 9: Feature Engineering — Has Collaborator (binary)

Trials with external collaborators (e.g., pharma sponsor + academic centre) typically  
involve more complex data flows and may exhibit different query rate patterns.

In [14]:
df["has_collaborator"] = (
    df["Collaborators"].notna()
    & (df["Collaborators"].astype(str).str.strip() != "")
).astype(int)

print("has_collaborator — value counts:")
print(df["has_collaborator"].value_counts())

has_collaborator — value counts:
has_collaborator
0    29826
1    20210
Name: count, dtype: int64


## Step 10: Feature Engineering — Study Design Sub-features

The `Study Design` column is a pipe-separated string containing  
Allocation, Intervention Model, Masking, and Primary Purpose.  
These are extracted as separate, ML-ready categorical features.

Masking is simplified to a numeric `blinding_level` score:  
- `NONE` → 0  
- `SINGLE` → 1  
- `DOUBLE` → 2  
- `TRIPLE` → 3  
- `QUADRUPLE` → 4

In [15]:
def extract_design_field(design_val, field_prefix):
    """Extract value for a given field prefix from the Study Design string."""
    if pd.isna(design_val):
        return "UNKNOWN"
    for part in str(design_val).split("|"):
        part = part.strip()
        if part.startswith(field_prefix):
            return part.replace(field_prefix, "").strip()
    return "UNKNOWN"


def simplify_masking(masking_val):
    """Convert masking description to a numeric blinding level (0-4)."""
    if pd.isna(masking_val):
        return 0
    m = str(masking_val).upper()
    if "QUADRUPLE" in m:
        return 4
    if "TRIPLE" in m:
        return 3
    if "DOUBLE" in m:
        return 2
    if "SINGLE" in m:
        return 1
    return 0  # NONE or UNKNOWN


# Extract Allocation
df["Allocation"] = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Allocation:")
)

# Extract Primary Purpose
df["Primary Purpose"] = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Primary Purpose:")
)

# Extract Masking and convert to numeric blinding level
masking_raw = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Masking:")
)
df["blinding_level"] = masking_raw.apply(simplify_masking)

print("Allocation — value counts:")
print(df["Allocation"].value_counts())
print()
print("Primary Purpose — value counts:")
print(df["Primary Purpose"].value_counts())
print()
print("blinding_level — value counts:")
print(df["blinding_level"].value_counts().sort_index())

Allocation — value counts:
Allocation
NA                20149
RANDOMIZED        18563
NON_RANDOMIZED     9482
UNKNOWN            1842
Name: count, dtype: int64

Primary Purpose — value counts:
Primary Purpose
TREATMENT                   37403
SUPPORTIVE_CARE              3659
DIAGNOSTIC                   2758
PREVENTION                   2514
OTHER                        1271
HEALTH_SERVICES_RESEARCH      687
SCREENING                     646
BASIC_SCIENCE                 553
UNKNOWN                       424
DEVICE_FEASIBILITY            115
ECT                             6
Name: count, dtype: int64

blinding_level — value counts:
blinding_level
0    41789
1     2757
2     2583
3     1199
4     1708
Name: count, dtype: int64


## Step 11: Standardise Phases

Some trials list multiple phases (e.g., `PHASE1|PHASE2`). The highest phase is kept.  
Missing phase values are filled with `NOT_REPORTED`.

In [16]:
PHASE_ORDER = ["PHASE4", "PHASE3", "PHASE2", "PHASE1", "EARLY_PHASE1"]

def clean_phase(phase_val):
    """Return the highest phase from a pipe-separated phase string."""
    if pd.isna(phase_val):
        return "NOT_REPORTED"
    phase_str = str(phase_val).upper().strip().replace(" ", "")
    if phase_str in ["", "NA", "N/A", "NAN"]:
        return "NOT_REPORTED"
    if "|" in phase_str:
        parts = [p.strip() for p in phase_str.split("|")]
        for candidate in PHASE_ORDER:
            if any(candidate in p for p in parts):
                return candidate
        return parts[0]  # fallback to first item
    return phase_str

df["Phases"] = df["Phases"].apply(clean_phase)

print("Phases — value counts after standardisation:")
print(df["Phases"].value_counts())

Phases — value counts after standardisation:
Phases
PHASE2          19846
NOT_REPORTED    12320
PHASE1          10199
PHASE3           5530
PHASE4           1387
EARLY_PHASE1      754
Name: count, dtype: int64


## Step 12: Impute Missing Enrollment by Phase Median

Trials with null enrollment are imputed using the median enrollment for  
their trial phase. This preserves realistic enrollment distributions  
by phase (Phase 3 trials typically enroll far more participants than Phase 1).  
A global median is used as a last-resort fallback.

In [17]:
print(f"Null enrollment before imputation: {df['Enrollment'].isna().sum():,}")

# Impute by phase median
phase_medians = df.groupby("Phases")["Enrollment"].transform("median")
df["Enrollment"] = df["Enrollment"].fillna(phase_medians)

# Global median fallback for any remaining nulls
df["Enrollment"] = df["Enrollment"].fillna(df["Enrollment"].median())

print(f"Null enrollment after imputation : {df['Enrollment'].isna().sum():,}")
print()
print("Phase median enrollment values:")
print(df.groupby("Phases")["Enrollment"].median().sort_values(ascending=False).round(0))

Null enrollment before imputation: 1,109
Null enrollment after imputation : 0

Phase median enrollment values:
Phases
PHASE3          248.0
PHASE4           65.0
NOT_REPORTED     60.0
PHASE2           40.0
PHASE1           24.0
EARLY_PHASE1     15.0
Name: Enrollment, dtype: float64


## Step 13: Select Final Columns and Set Index

In [18]:
# These are the final columns kept for ML work
FINAL_COLUMNS = [
    "NCT Number",           # Unique trial identifier (will become index)
    "Study Status",         # COMPLETED or TERMINATED
    "Study Results",        # YES / NO — whether results were reported
    "Conditions",           # Therapeutic area / disease
    "Sponsor",              # Sponsoring organisation
    "Sex",                  # Participant sex eligibility
    "Age",                  # Age group eligibility
    "Phases",               # Trial phase (standardised)
    "Enrollment",           # Number of participants (imputed)
    "Trial Duration (days)",# Engineered: duration in days
    "Number of Sites",      # Engineered: count of trial sites
    "Intervention Type",    # Engineered: DRUG / DEVICE / BIOLOGICAL etc.
    "has_collaborator",     # Engineered: binary flag
    "Allocation",           # Engineered: RANDOMIZED / NON_RANDOMIZED / NA
    "Primary Purpose",      # Engineered: TREATMENT / DIAGNOSTIC etc.
    "blinding_level",       # Engineered: numeric masking score (0-4)
]

# Keep only columns that exist (safety check)
final_cols = [c for c in FINAL_COLUMNS if c in df.columns]
df_final = df[final_cols].copy()

# Set NCT Number as the index
if "NCT Number" in df_final.columns:
    df_final = df_final.set_index("NCT Number")

print(f"Final dataset shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print()
print("Final columns:")
for col in df_final.columns:
    print(f"  - {col}")

Final dataset shape: 50,036 rows × 15 columns

Final columns:
  - Study Status
  - Study Results
  - Conditions
  - Sponsor
  - Sex
  - Age
  - Phases
  - Enrollment
  - Trial Duration (days)
  - Number of Sites
  - Intervention Type
  - has_collaborator
  - Allocation
  - Primary Purpose
  - blinding_level


## Step 14: Final Quality Checks

In [19]:
print("=== FINAL NULL COUNTS ===")
null_final = df_final.isnull().sum()
print(null_final[null_final > 0] if null_final.any() else "No nulls remaining.")
print()

print("=== DTYPES ===")
print(df_final.dtypes)
print()

print("=== STUDY STATUS DISTRIBUTION ===")
print(df_final["Study Status"].value_counts())
print()

print("=== ENROLLMENT — OUTLIER CHECK ===")
print(df_final["Enrollment"].describe().round(1))
large = df_final[df_final["Enrollment"] > 10000]
print(f"Trials with enrollment > 10,000: {len(large):,}")

print()
print("=== TRIAL DURATION — OUTLIER CHECK ===")
print(df_final["Trial Duration (days)"].describe().round(1))

=== FINAL NULL COUNTS ===
Sex    16
dtype: int64

=== DTYPES ===
Study Status              object
Study Results             object
Conditions                object
Sponsor                   object
Sex                       object
Age                       object
Phases                    object
Enrollment               float64
Trial Duration (days)      int64
Number of Sites            int64
Intervention Type         object
has_collaborator           int32
Allocation                object
Primary Purpose           object
blinding_level             int64
dtype: object

=== STUDY STATUS DISTRIBUTION ===
Study Status
COMPLETED     40366
TERMINATED     9670
Name: count, dtype: int64

=== ENROLLMENT — OUTLIER CHECK ===
count     50036.0
mean        244.6
std        3343.4
min           1.0
25%          20.0
50%          42.0
75%         100.0
max      400415.0
Name: Enrollment, dtype: float64
Trials with enrollment > 10,000: 128

=== TRIAL DURATION — OUTLIER CHECK ===
count    50036.0
mean 

In [20]:
# Preview the final cleaned dataset
print("=== FIRST 5 ROWS OF CLEANED DATA ===")
df_final.head()

=== FIRST 5 ROWS OF CLEANED DATA ===


,Study Status,Study Results,Conditions,Sponsor,Sex,Age,Phases,Enrollment,Trial Duration (days),Number of Sites,Intervention Type,has_collaborator,Allocation,Primary Purpose,blinding_level
NCT Number,,,,,,,,,,,,,,,
NCT00768222,COMPLETED,YES,Breast Cancer,"Ethicon, Inc.",ALL,"ADULT, OLDER_ADULT",PHASE4,101.0,212,6,DEVICE,1,RANDOMIZED,TREATMENT,0
NCT03471351,TERMINATED,NO,Classical Hodgkin Lymphoma,Rhizen Pharmaceuticals SA,ALL,"ADULT, OLDER_ADULT",PHASE1,2.0,210,3,DRUG,0,NA,TREATMENT,0
NCT00498251,COMPLETED,NO,"Lung Injury, Acute|Thoracotomy|Anesthesia|Intensive Care...","University Hospital, Geneva",ALL,"CHILD, ADULT, OLDER_ADULT",NOT_REPORTED,30.0,1003,1,DRUG,0,RANDOMIZED,TREATMENT,2
NCT00275951,COMPLETED,NO,Gastric Cancer,National Taiwan University Hospital,ALL,"ADULT, OLDER_ADULT",PHASE2,39.0,1278,1,DRUG,1,NON_RANDOMIZED,TREATMENT,0
NCT02025231,TERMINATED,NO,Malignant Glioma|Glioblastoma,"Rigshospitalet, Denmark",ALL,"ADULT, OLDER_ADULT",PHASE2,31.0,1217,2,RADIATION,1,NA,TREATMENT,0


## Step 15: Save Cleaned Dataset

In [21]:
df_final.to_csv(OUTPUT_FILE)

print(f"Cleaned dataset saved to: {OUTPUT_FILE}")
print(f"Final shape             : {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print()
print("=== CLEANING PIPELINE SUMMARY ===")
print(f"  Raw dataset rows          : {len(df_raw):,}")
print(f"  After INTERVENTIONAL only : {len(df_raw[df_raw['Study Type'].str.upper().str.strip() == 'INTERVENTIONAL']):,}")
print(f"  Final cleaned rows        : {df_final.shape[0]:,}")
print(f"  Final feature columns     : {df_final.shape[1]}")
print()
print("Ready for target variable engineering and ML model training.")

Cleaned dataset saved to: C:/Users/Puni/Desktop/Thesis/cleaned_clinical_trials.csv
Final shape             : 50,036 rows × 15 columns

=== CLEANING PIPELINE SUMMARY ===
  Raw dataset rows          : 119,151
  After INTERVENTIONAL only : 94,006
  Final cleaned rows        : 50,036
  Final feature columns     : 15

Ready for target variable engineering and ML model training.
